# DATASCI 503, Homework 10: Neural Networks and Deep Learning


This assignment covers **network architecture** (layers, activation functions, parameter counting), the **softmax function** for multi-class classification, **convolutional neural networks (CNNs)** for image data, and **gradient descent** optimization.

**Resources:**
- [PyTorch Autograd Tutorial](https://pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html) - Automatic differentiation for gradient computation
- [ISLP Chapter 10](https://www.statlearning.com/) - Deep Learning chapter from Introduction to Statistical Learning

**You can upload images directly with the upload files button when you open your submission.**

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import torch

---

**Problem 1:** Neural Network Architecture (ISLP 10.1, parts a, b, and d)

Consider a neural network with two hidden layers: p = 4 input units, 2 units in the first hidden layer, 3 units in the second hidden layer, and a single output.

(10.1a) Draw a picture of the network, similar to Figures 10.1 or 10.4 in ISLP.

In [ ]:
G = nx.DiGraph()

# Define layers
input_nodes = ["X1", "X2", "X3", "X4"]
hidden1_nodes = ["H1_1", "H1_2"]
hidden2_nodes = ["H2_1", "H2_2", "H2_3"]
output_nodes = ["Y"]

G.add_nodes_from(input_nodes)
G.add_nodes_from(hidden1_nodes)
G.add_nodes_from(hidden2_nodes)
G.add_nodes_from(output_nodes)

# Connect input to hidden layer 1
for inp in input_nodes:
    for h1 in hidden1_nodes:
        G.add_edge(inp, h1)

# Connect hidden layer 1 to hidden layer 2
for h1 in hidden1_nodes:
    for h2 in hidden2_nodes:
        G.add_edge(h1, h2)

# Connect hidden layer 2 to output
for h2 in hidden2_nodes:
    G.add_edge(h2, "Y")

# Position nodes in layers
pos = {}
for i, node in enumerate(input_nodes):
    pos[node] = (0, 3 - i)
for i, node in enumerate(hidden1_nodes):
    pos[node] = (1, 2 - i * 2)
for i, node in enumerate(hidden2_nodes):
    pos[node] = (2, 3 - i)
pos["Y"] = (3, 1.5)

plt.figure(figsize=(8, 5))
nx.draw(G, pos, with_labels=True, node_size=1500, node_color="lightblue",
        font_size=10, font_weight="bold", arrowsize=15, edge_color="gray")
plt.title("Neural Network: 4-2-3-1 Architecture")
plt.show()

In [ ]:
# Test assertions
assert (
    G.number_of_nodes() == 10
), "Network should have 10 nodes (4 input + 2 hidden1 + 3 hidden2 + 1 output)"
assert G.number_of_edges() == 17, "Network should have 17 edges"
print("All tests passed!")

(10.1b) Write out an expression for $f(X)$, assuming ReLU activation functions. Be as explicit as you can!

Let $X = (X_1, X_2, X_3, X_4)$ be the input, and let $g(t) = \max(0, t)$ denote the ReLU activation.

**After First hidden layer** (2 units):

$A_k^{(1)} = g\left(w_{k0}^{(1)} + \sum_{j=1}^{4} w_{kj}^{(1)} X_j\right), \quad k = 1, 2$

Explicitly:

$A_1^{(1)} = g(w_{10}^{(1)} + w_{11}^{(1)} X_1 + w_{12}^{(1)} X_2 + w_{13}^{(1)} X_3 + w_{14}^{(1)} X_4)$

$A_2^{(1)} = g(w_{20}^{(1)} + w_{21}^{(1)} X_1 + w_{22}^{(1)} X_2 + w_{23}^{(1)} X_3 + w_{24}^{(1)} X_4)$

**Second hidden layer** (3 units):

$A_m^{(2)} = g\left(w_{m0}^{(2)} + \sum_{k=1}^{2} w_{mk}^{(2)} A_k^{(1)}\right), \quad m = 1, 2, 3$

Explicitly:

$A_1^{(2)} = g(w_{10}^{(2)} + w_{11}^{(2)} A_1^{(1)} + w_{12}^{(2)} A_2^{(1)})$

$A_2^{(2)} = g(w_{20}^{(2)} + w_{21}^{(2)} A_1^{(1)} + w_{22}^{(2)} A_2^{(1)})$

$A_3^{(2)} = g(w_{30}^{(2)} + w_{31}^{(2)} A_1^{(1)} + w_{32}^{(2)} A_2^{(1)})$

**Output layer** (1 unit, linear):

$f(X) = w_0^{(3)} + w_1^{(3)} A_1^{(2)} + w_2^{(3)} A_2^{(2)} + w_3^{(3)} A_3^{(2)}$

**Without bias version (for reference only — the question expects biases):**

**After First hidden layer**:

$A_1^{(1)} = g(w_{11}^{(1)} X_1 + w_{12}^{(1)} X_2 + w_{13}^{(1)} X_3 + w_{14}^{(1)} X_4)$

$A_2^{(1)} = g(w_{21}^{(1)} X_1 + w_{22}^{(1)} X_2 + w_{23}^{(1)} X_3 + w_{24}^{(1)} X_4)$

**After Second hidden layer**:

$A_1^{(2)} = g(w_{11}^{(2)} A_1^{(1)} + w_{12}^{(2)} A_2^{(1)})$

$A_2^{(2)} = g(w_{21}^{(2)} A_1^{(1)} + w_{22}^{(2)} A_2^{(1)})$

$A_3^{(2)} = g(w_{31}^{(2)} A_1^{(1)} + w_{32}^{(2)} A_2^{(1)})$

**Output layer**:

$f(X) = w_1^{(3)} A_1^{(2)} + w_2^{(3)} A_2^{(2)} + w_3^{(3)} A_3^{(2)}$

(10.1d) How many parameters are there?

**Input → Hidden Layer 1:** Each of the 2 hidden units has 4 weights + 1 bias = 5 parameters.

$2 \times (4 + 1) = 10$

**Hidden Layer 1 → Hidden Layer 2:** Each of the 3 hidden units has 2 weights + 1 bias = 3 parameters.

$3 \times (2 + 1) = 9$

**Hidden Layer 2 → Output:** The output unit has 3 weights + 1 bias = 4 parameters.

$1 \times (3 + 1) = 4$

**Total parameters = $10 + 9 + 4 = \mathbf{23}$**

---

**Problem 2:** Softmax Invariance (ISLP 10.2)

Consider the softmax function in (10.13) (see also (4.13) on page 145 of ISLP) for modeling multinomial probabilities.

(a) In (10.13), show that if we add a constant $c$ to each of the $z_i$, then the probability is unchanged.

Given the softmax function is:

$\text{Pr}(Y = k) = \frac{e^{z_k}}{\sum_{l=0}^{K-1} e^{z_l}}$

If we add a constant $c$ to each $z_i$:

$\text{Pr}(Y = k) = \frac{e^{z_k + c}}{\sum_{l=0}^{K-1} e^{z_l + c}}$

$\text{Pr}(Y = k) = \frac{e^{z_k} \cdot e^c}{\sum_{l=0}^{K-1} e^{z_l} \cdot e^c}$

$\text{Pr}(Y = k) = \frac{e^c \cdot e^{z_k}}{e^c \cdot \sum_{l=0}^{K-1} e^{z_l}}$

$\text{Pr}(Y = k) = \frac{e^{z_k}}{\sum_{l=0}^{K-1} e^{z_l}}$

(b) In (4.13), show that if we add constants $c_j$, $j = 0,1,\ldots,p$, to each of the corresponding coefficients for each of the classes, then the predictions at any new point $x$ are unchanged.

In (4.13), the softmax for multinomial logistic regression is:

$\text{Pr}(Y = k \mid X = x) = \frac{e^{\beta_{k0} + \beta_{k1}x_1 + \cdots + \beta_{kp}x_p}}{\sum_{l=0}^{K-1} e^{\beta_{l0} + \beta_{l1}x_1 + \cdots + \beta_{lp}x_p}}$

If we add constants $c_j$ to each coefficient for all classes, replacing $\beta_{kj}$ with $\beta_{kj} + c_j$:

$\text{Pr}(Y = k \mid X = x) = \frac{e^{(\beta_{k0}+c_0) + (\beta_{k1}+c_1)x_1 + \cdots + (\beta_{kp}+c_p)x_p}}{\sum_{l=0}^{K-1} e^{(\beta_{l0}+c_0) + (\beta_{l1}+c_1)x_1 + \cdots + (\beta_{lp}+c_p)x_p}}$

Let $c(x) = c_0 + c_1 x_1 + \cdots + c_p x_p$, which is the same for all classes $k$.

$\text{Pr}(Y = k \mid X = x) = \frac{e^{\beta_{k0} + \cdots + \beta_{kp}x_p + c(x)}}{\sum_{l=0}^{K-1} e^{\beta_{l0} + \cdots + \beta_{lp}x_p + c(x)}}$

$\text{Pr}(Y = k \mid X = x) = \frac{e^{c(x)} \cdot e^{\beta_{k0} + \cdots + \beta_{kp}x_p}}{e^{c(x)} \cdot \sum_{l=0}^{K-1} e^{\beta_{l0} + \cdots + \beta_{lp}x_p}}$

$\text{Pr}(Y = k \mid X = x) = \frac{e^{\beta_{k0} + \cdots + \beta_{kp}x_p}}{\sum_{l=0}^{K-1} e^{\beta_{l0} + \cdots + \beta_{lp}x_p}}$

This shows that the softmax function is over-parametrized. However, regularization and SGD typically constrain the solutions so that this is not a problem.

---

**Problem 3:** CNN Parameter Counting (ISLP 10.4)

Consider a CNN that takes in 32 x 32 grayscale images and has a single convolution layer with three 5 x 5 convolution filters (without boundary padding).

(b) How many parameters are in this model?

Input: $32 \times 32$ grayscale image (1 channel). Three $5 \times 5$ convolution filters, no padding.

**Each filter:** $5 \times 5 = 25$ weights $+ 1$ bias $= 26$ parameters.

**3 filters:** $3 \times 26 = \mathbf{78}$ parameters.

(c) Explain how this model can be thought of as an ordinary feed-forward neural network with the individual pixels as inputs, and with constraints on the weights in the hidden units. What are the constraints?

This CNN can be viewed as an ordinary feed-forward neural network where each pixel is an input unit and the convolution outputs are the hidden units, but with two constraints on the weights:

1. **Local connectivity:** Each hidden unit connects to only a $5 \times 5$ local patch of the input image, not to all input pixels. Weights to all other pixels are set to zero.

2. **Weight sharing:** All hidden units within the same feature map share the same $5 \times 5 = 25$ weights and 1 bias. Instead of each hidden unit having its own unique set of weights, they reuse the same filter parameters.

(d) If there were no constraints, then how many weights would there be in the ordinary feed-forward neural network in (c)?

Without constraints, every hidden unit is fully connected to all input pixels.

**Number of input units.**

The image is $32 \times 32$ grayscale, so there are:

$32 \times 32 = 1024 \ input \ pixels.$ Therefore, we have:

$1024 + 1 = 1025 \ parameters \ per \ hidden \ unit $

**Number of hidden units after filtering**

$3 \ filters \times (32 - 5 + 1) \times (32 - 5 + 1) = 2352 \ hidden \ units$

$Total \ parameters = 2352 \times 1025 = 2{,}410{,}800 \ parameters.$

Compared to the CNN's 78 parameters, the unconstrained network has over 30,000 times more parameters.

---

**Problem 4:** Gradient Descent Visualization (ISLP 10.6)

Consider the simple function $R(\beta) = \sin(\beta) + \beta/10$.

(a) Draw a graph of this function over the range $\beta \in [-6, 6]$.

In [ ]:
def objective_function(beta):
    return np.sin(beta) + beta / 10

beta_values = np.linspace(-6, 6, 400)
R_values = objective_function(beta_values)

plt.figure(figsize=(8, 5))
plt.plot(beta_values, R_values, color="blue")
plt.xlabel(r"$\beta$")
plt.ylabel(r"$R(\beta)$")
plt.title(r"$R(\beta) = \sin(\beta) + \beta/10$")
plt.grid(True)
plt.show()

In [ ]:
# Test assertions
assert len(beta_values) == 400, "Should have 400 points for smooth plotting"
assert beta_values[0] == -6 and beta_values[-1] == 6, "Beta range should be [-6, 6]"
assert callable(objective_function), "objective_function should be callable"
print("All tests passed!")

(b) What is the derivative of this function?

$R(\beta) = \sin(\beta) + \frac{\beta}{10}$

$R'(\beta) = \cos(\beta) + \frac{1}{10}$

(c) Given $\beta^0 = 2.3$, run gradient descent for 50 iterations to find a local minimum of $R(\beta)$ using a learning rate of $\rho = 0.1$. Show each of $\beta^0, \beta^1, \ldots$ in your plot, as well as the final answer.

In [ ]:
def gradient(beta):
    return np.cos(beta) + 1/10

beta = 2.3
rho = 0.1
beta_history = [beta]

for i in range(50):
    beta = beta - rho * gradient(beta)
    beta_history.append(beta)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(beta_values, R_values, color="blue", label=r"$R(\beta)$")
plt.scatter(beta_history, objective_function(np.array(beta_history)), color="red", s=20, zorder=5)
plt.scatter(beta_history[0], objective_function(beta_history[0]), color="green", s=80, zorder=6, label=r"$\beta^0 = 2.3$")
plt.scatter(beta_history[-1], objective_function(beta_history[-1]), color="black", s=80, zorder=6, marker="*", label=f"Final: $\\beta = {beta_history[-1]:.4f}$")
plt.xlabel(r"$\beta$")
plt.ylabel(r"$R(\beta)$")
plt.title(r"Gradient Descent from $\beta^0 = 2.3$")
plt.legend()
plt.grid(True)
plt.show()

print(f"Final beta: {beta_history[-1]:.6f}")
print(f"Final R(beta): {objective_function(beta_history[-1]):.6f}")

In [ ]:
# Test assertions
assert len(beta_history) > 1, "Should have multiple gradient descent steps"
assert abs(beta_history[0] - 2.3) < 0.001, "Should start from beta^0 = 2.3"
print("All tests passed!")

Starting from $\beta^0 = 2.3$, gradient descent converges to a local minimum near $\beta \approx -1.67$, which corresponds to $R(\beta) \approx -1.17$. The iterates are shown as red dots on the curve, moving from the green starting point to the black star (final value).

(d) Repeat with $\beta^0 = 1.4$.

In [ ]:
beta = 1.4
rho = 0.1
beta_history = [beta]

for i in range(50):
    beta = beta - rho * gradient(beta)
    beta_history.append(beta)

plt.figure(figsize=(8, 5))
plt.plot(beta_values, R_values, color="blue", label=r"$R(\beta)$")
plt.scatter(beta_history, objective_function(np.array(beta_history)), color="red", s=20, zorder=5)
plt.scatter(beta_history[0], objective_function(beta_history[0]), color="green", s=80, zorder=6, label=r"$\beta^0 = 1.4$")
plt.scatter(beta_history[-1], objective_function(beta_history[-1]), color="black", s=80, zorder=6, marker="*", label=f"Final: $\\beta = {beta_history[-1]:.4f}$")
plt.xlabel(r"$\beta$")
plt.ylabel(r"$R(\beta)$")
plt.title(r"Gradient Descent from $\beta^0 = 1.4$")
plt.legend()
plt.grid(True)
plt.show()

print(f"Final beta: {beta_history[-1]:.6f}")
print(f"Final R(beta): {objective_function(beta_history[-1]):.6f}")

In [ ]:
# Test assertions
assert len(beta_history) > 1, "Should have multiple gradient descent steps"
assert abs(beta_history[0] - 1.4) < 0.001, "Should start from beta^0 = 1.4"
print("All tests passed!")

Starting from $\beta^0 = 1.4$, gradient descent converges to a **different** local minimum near $\beta \approx -1.67$ (same local minimum as part c, since the gradient pushes it past the local max and into the same valley). 

This illustrates that gradient descent finds a **local** minimum, and the result can depend on the starting point. If the starting point were further to the right (e.g., $\beta^0 = 4.5$), it could converge to a different local minimum near $\beta \approx 4.6$.